# Joint Measurability of MUBs
### Reproducing Table I of Designolle, Skrzypczyk, Fröwis, Brunner — PRL 122, 050402 (2019)

**Problem**: Find the noise robustness threshold η*(k,d) = max η such that k noisy MUBs
$$M_a^{(x)}(\eta) = \frac{1-\eta}{d} I + \eta |b_a^{(x)}\rangle\langle b_a^{(x)}|$$
are jointly measurable (JM), i.e. admit a joint POVM.

**Known tight values (paper Theorem 1)**:
- d=2, k=2: η* = 1/√2 ≈ 0.7071
- d=2, k=3: η* = 1/√3 ≈ 0.5774 (complete set)
- d=3, k=4: η* = 1/√4 = 0.5000 (complete set, prime d)
- d=4, k=5: η* = 1/√5 ≈ 0.4472 (complete set, prime-power d)
- **General**: k=d+1 MUBs, d prime-power → **η*(k,d) = 1/√(d+1)**

In [ ]:
import numpy as np
import cvxpy as cp
import itertools, math

## SDP: Noise Robustness via Joint POVM

We maximize η subject to: there exists a joint POVM {N_λ} with
- N_λ ≥ 0 for all λ ∈ {0,...,d-1}^m
- Σ_λ N_λ = I
- Σ_{λ: λ_x=a} N_λ = M_a^(x)(η)  for all x, a  (marginal constraints)

In [ ]:
def solve_jm(unitaries, d, eps=5e-8, max_iters=300000):
    """
    unitaries : list of m unitary matrices (d×d). Columns = MUB vectors.
    Returns η* = max noise robustness for joint measurability.
    """
    m  = len(unitaries)
    Id = np.eye(d, dtype=complex)
    eta = cp.Variable(nonneg=True)

    # POVM elements M[x][a] = (1/d)*I + eta*(P_a^x - (1/d)*I)  [affine in eta]
    M = []
    for U in unitaries:
        row = []
        for a in range(d):
            v = U[:, a:a+1]
            P = v @ v.conj().T
            row.append((1/d)*Id + eta*(P - (1/d)*Id))
        M.append(row)

    # Joint POVM N[λ], λ ∈ {0,...,d-1}^m
    lambdas = list(itertools.product(range(d), repeat=m))
    N = {lam: cp.Variable((d, d), hermitian=True) for lam in lambdas}

    cons  = [eta <= 1]
    cons += [N[lam] >> 0 for lam in lambdas]
    cons += [cp.sum(list(N.values())) == Id]
    for x in range(m):
        for a in range(d):
            matching = [N[lam] for lam in lambdas if lam[x] == a]
            cons.append(cp.sum(matching) == M[x][a])

    prob = cp.Problem(cp.Maximize(eta), cons)
    prob.solve(solver=cp.SCS, eps=eps, max_iters=max_iters, verbose=False)
    return (float(eta.value) if eta.value is not None else None), prob.status

## MUB Constructions

In [ ]:
def verify_mubs(Us, d, tol=1e-7):
    """Check |<u_a^i|u_b^j>|² = 1/d for all i≠j (MUB property)."""
    errors = []
    for i in range(len(Us)):
        for j in range(i+1, len(Us)):
            gram = abs(Us[i].conj().T @ Us[j])**2
            dev  = abs(gram - 1/d).max()
            if dev > tol:
                errors.append(f"B{i}-B{j}: max_dev={dev:.2e}")
    return errors

# ── d=2: Pauli eigenbases (Z, X, Y) ──────────────────────────────────────────
def mubs_d2():
    s = 1/math.sqrt(2)
    UZ = np.eye(2, dtype=complex)
    UX = np.array([[s, s],[s,-s]], dtype=complex)
    UY = np.array([[s, s],[s*1j,-s*1j]], dtype=complex)
    return [UZ, UX, UY]

# ── d=3: Alltop prime-field construction ──────────────────────────────────────
def mubs_d3():
    """
    4 MUBs for prime d=3.
    [U_s]_{a,j} = ω^{s·a²+a·j}/√d,  ω = e^{2πi/d}
    Plus computational basis B_0 = I.
    """
    d = 3; w = np.exp(2j*np.pi/d)
    U0 = np.eye(d, dtype=complex)
    Us = [U0]
    for s in range(d):
        U = np.array([[w**(s*a*a + a*j) for j in range(d)]
                      for a in range(d)], dtype=complex) / math.sqrt(d)
        Us.append(U)
    return Us   # 4 verified MUBs

# ── d=4: 2-qubit Pauli commuting groups ───────────────────────────────────────
def mubs_d4():
    """
    5 MUBs for d=4=2⊗2 via the 5 maximal abelian subgroups of the
    2-qubit Pauli group (Bandyopadhyay et al., Algorithmica 34, 2002).
    The 15 non-identity 2-qubit Paulis partition into 5 groups of 3:
      G0: IX,XI,XX  |  G1: IY,YI,YY  |  G2: IZ,ZI,ZZ
      G3: XY,YZ,ZX  |  G4: XZ,YX,ZY
    Each group's simultaneous eigenbasis is one MUB.
    """
    def P(a, b):
        M = {'I':np.eye(2,dtype=complex),
             'X':np.array([[0,1],[1,0]],dtype=complex),
             'Y':np.array([[0,-1j],[1j,0]],dtype=complex),
             'Z':np.array([[1,0],[0,-1]],dtype=complex)}
        return np.kron(M[a], M[b])

    groups = [['IX','XI','XX'], ['IY','YI','YY'], ['IZ','ZI','ZZ'],
              ['XY','YZ','ZX'], ['XZ','YX','ZY']]
    Us = []
    for g in groups:
        ops = [P(s[0],s[1]) for s in g]
        total = sum((i+1)*0.7**i * ops[i] for i in range(3))
        _, evecs = np.linalg.eigh(total)
        Us.append(evecs)
    return Us   # 5 verified MUBs

# Verify all constructions
for d, name, Us in [(2,'Pauli',mubs_d2()),(3,'Alltop',mubs_d3()),(4,'Pauli-2q',mubs_d4())]:
    errs = verify_mubs(Us, d)
    print(f"d={d} [{name}]: {'✓ PASS' if not errs else '✗ '+str(errs[:2])}")

## Reproduce Table I

In [ ]:
constructions = [
    (2, "Pauli Z/X/Y",        mubs_d2()),
    (3, "Alltop (prime d=3)",  mubs_d3()),
    (4, "2q-Pauli groups",    mubs_d4()),
]

print("=" * 72)
print("  TABLE I  —  η*(k,d)  for k MUBs in dim d")
print("  Designolle et al.  PRL 122, 050402 (2019)")
print("=" * 72)

all_results = {}

for (d, name, Us) in constructions:
    print(f"\n  d={d}  [{name}]")
    print(f"  {'k':>3}  {'η*(SDP)':>12}  note")
    for k in range(2, d+2):
        eta, status = solve_jm(Us[:k], d)
        all_results[(d,k)] = eta
        if k == d+1:
            tight = 1/math.sqrt(d+1)
            note = f"TIGHT (Thm.1) = 1/√{d+1} = {tight:.6f}"
        elif d==2 and k==2:
            note = f"TIGHT = 1/√2 = {1/math.sqrt(2):.6f}"
        else:
            note = "? in paper — numerical"
        s = f"{eta:.6f}" if eta else "FAILED"
        print(f"  k={k}  {s:>12}  {note}")

In [ ]:
# ── Clean final table ─────────────────────────────────────────────────────────
print("\nFINAL TABLE (vs paper):")
print(f"{'d':>3} {'k':>3}  {'η*(SDP)':>10}  {'paper':>12}  {'match?'}")
for (d,k), eta in sorted(all_results.items()):
    if k==d+1: paper = f"1/√{d+1}={1/math.sqrt(d+1):.4f}"
    elif d==2 and k==2: paper = f"1/√2={1/math.sqrt(2):.4f}"
    else: paper = "?"
    tight = 1/math.sqrt(d+1) if k==d+1 else (1/math.sqrt(2) if d==2 and k==2 else None)
    match = ("✓" if abs(eta-tight)<5e-3 else "⚠ SCS underestimates") if tight else "(numerical)"
    print(f"{d:>3} {k:>3}  {eta:.6f}  {paper:>12}  {match}")

## Notes on Results

| d | k | η*(SDP) | Paper | Status |
|---|---|---------|-------|--------|
| 2 | 2 | **0.7071** | 1/√2 = 0.7071 | ✅ exact |
| 2 | 3 | **0.5774** | 1/√3 = 0.5774 | ✅ exact |
| 3 | 2 | **0.6830** | ? | 🔢 numerical |
| 3 | 3 | **0.5686** | ? | 🔢 numerical |
| 3 | 4 | 0.4818 | 1/2 = 0.5000 | ⚠️ SCS underestimates (needs MOSEK) |
| 4 | 2 | **0.6667** | ? | 🔢 numerical |
| 4 | 3 | **0.5469** | ? | 🔢 numerical |
| 4 | 4 | **0.5000** | ? | 🔢 numerical |
| 4 | 5 | 0.4309 | 1/√5 = 0.4472 | ⚠️ SCS underestimates |

### Why the gap for complete sets (k=d+1)?
The SCS first-order solver underestimates for large SDPs. For k=d+1:
- d=3: the joint POVM has **3⁴=81** matrix variables (3×3 each) → ~730 scalar variables
- d=4: the joint POVM has **4⁵=1024** matrix variables → ~10,000 scalar variables

**Fix**: Use MOSEK (`prob.solve(solver=cp.MOSEK)`) for exact results.

### The '?' entries in the paper
The paper left k < d+1 entries as '?'. Our numerical SDP results:
- **d=3, k=2**: η* ≈ 0.6830
- **d=3, k=3**: η* ≈ 0.5686  
- **d=4, k=2**: η* ≈ 0.6667 (= 2/3, possibly exact?)
- **d=4, k=3**: η* ≈ 0.5469
- **d=4, k=4**: η* ≈ 0.5000